<a href="https://colab.research.google.com/github/ancestor9/2026_Fall_Application-Deployment/blob/main/CS/02_Caching_hit_miss.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

LRU (Least Recently Used) 알고리즘:

제한된 캐시 공간이 가득 찼을 때 '가장 오래전에 사용된 데이터'를 제거하고 새로운 데이터를 채우는 방식

### cache_info() 함수:

- hits: 캐시에 저장된 값을 재사용하여 성공적으로 불러온 횟수

- misses: 캐시에 값이 없어 실제 함수를 실행한 횟수

- maxsize: 최대 캐시 보관 개수

- currsize: 현재 캐시에 담겨 있는 개수

In [10]:
import time
from functools import lru_cache

# maxsize=3: 최근에 사용된 결과 최대 3개까지 캐시 공간에 보관 (LRU 적용)

@lru_cache(maxsize=3)
def heavy_calculation(n):

    """실행 시 3초가 소요되는 무거운 연산 함수"""
    print(f"  [Cache Miss!] heavy_calculation({n}) 실행 중... ({3}초 소요)")
    time.sleep(3)  # RAM 조회나 DB/네트워크 연산을 흉내 냄
    return n * 10

In [11]:
print("heavy_calculation(n)이 실제로 몇 초 걸리는지 측정합니다.")

# 캐시 초기화를 통해 항상 Cache Miss를 발생시켜 '무거운 연산' 시간을 측정
heavy_calculation.cache_clear()
print(f"캐시 초기화 후 캐시 상태: {heavy_calculation.cache_info()}")

heavy_calculation(n)이 실제로 몇 초 걸리는지 측정합니다.
캐시 초기화 후 캐시 상태: CacheInfo(hits=0, misses=0, maxsize=3, currsize=0)


In [12]:
# heavy_calculation(1) 호출 시간 측정 (Cache Miss 발생)
print("\n--- heavy_calculation(1) (Cache Miss) 호출 시간을 측정합니다 ---")
start_time = time.time()
heavy_calculation(1) # 이 호출은 5초가 소요됩니다.
end_time = time.time()
duration = end_time - start_time
print(f"heavy_calculation(1) 호출에 실제 {duration:.2f}초가 소요되었습니다.")
print(f"현재 캐시 상태: {heavy_calculation.cache_info()}")


--- heavy_calculation(1) (Cache Miss) 호출 시간을 측정합니다 ---
  [Cache Miss!] heavy_calculation(1) 실행 중... (3초 소요)
heavy_calculation(1) 호출에 실제 3.00초가 소요되었습니다.
현재 캐시 상태: CacheInfo(hits=0, misses=1, maxsize=3, currsize=1)


In [13]:
# heavy_calculation(1)을 다시 호출하여 캐시 히트 시 시간을 측정
print("\n--- heavy_calculation(1) (Cache Hit) 호출 시간을 측정합니다 ---")
start_time = time.time()
heavy_calculation(1) # 이 호출은 캐시 히트로 거의 시간이 소요되지 않음
end_time = time.time()
duration = end_time - start_time
print(f"heavy_calculation(1) (캐시 히트) 호출에 실제 {duration:.4f}초가 소요되었습니다.") # 캐시 히트는 매우 빠르므로 더 정밀하게 표시
print(f"현재 캐시 상태: {heavy_calculation.cache_info()}")


--- heavy_calculation(1) (Cache Hit) 호출 시간을 측정합니다 ---
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0001초가 소요되었습니다.
현재 캐시 상태: CacheInfo(hits=1, misses=1, maxsize=3, currsize=1)


In [16]:
print("=== 1. 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===")

for _ in range(10):
    print("\n1) heavy_calculation(1) 첫 번째 호출:")
    start_time = time.time()
    result = heavy_calculation(1)
    end_time = time.time()
    duration = end_time - start_time
    print(f"   캐시 상태: {heavy_calculation.cache_info()}")
    print(f"heavy_calculation(1) (캐시 히트) 호출에 실제 {duration:.4f}초가 소요되었습니다.")
    print('*'*100)

=== 1. 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=22, misses=1, maxsize=3, currsize=1)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=23, misses=1, maxsize=3, currsize=1)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=24, misses=1, maxsize=3, currsize=1)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=25, misses=1, maxsize=3, currsize=1)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
**********************************************************

- misses=1 : 함수인자로 1이 처음 전달되었을 때, 캐시에 값이 없었으므로 딱 1번만 실제 연산(1초 소요)을 수행하고 그 이후로는 인자 1이 캐시에 이미 보관되어 있으므로, 아무리 많이 호출해도 misses는 늘어나지 않고 항상 1에 고정

- currsize=1 : currsize는 현재 캐시 메모리에 실제로 저장되어 있는 서로 다른 결과값의 개수로 지금까지 캐시에 들어간 인자가 1 하나뿐이기 때문에, maxsize=3중 용량을 1개만 사용 중이라는 의미

In [17]:
print("\n=== 2. LRU (Least Recently Used) 캐시 교체 알고리즘 확인 ===")
# maxsize=3 이므로 3개까지 저장
print("\n- 추가 인자 호출: 2, 3")
heavy_calculation(2)
heavy_calculation(3)
print(f"   현재 캐시 상태 (1, 2, 3 저장됨): {heavy_calculation.cache_info()}")


=== 2. LRU (Least Recently Used) 캐시 교체 알고리즘 확인 ===

- 추가 인자 호출: 2, 3
  [Cache Miss!] heavy_calculation(2) 실행 중... (3초 소요)
  [Cache Miss!] heavy_calculation(3) 실행 중... (3초 소요)
   현재 캐시 상태 (1, 2, 3 저장됨): CacheInfo(hits=31, misses=3, maxsize=3, currsize=3)


In [18]:
# 4를 호출하면 maxsize(3)를 초과하므로 '가장 오래전에 사용된 1'이 캐시에서 삭제됨
print("\n- 새로운 인자 4 호출 (maxsize=3 초과 발생):")
heavy_calculation(4)
print(f"   현재 캐시 상태: {heavy_calculation.cache_info()}")


- 새로운 인자 4 호출 (maxsize=3 초과 발생):
  [Cache Miss!] heavy_calculation(4) 실행 중... (3초 소요)
   현재 캐시 상태: CacheInfo(hits=31, misses=4, maxsize=3, currsize=3)


In [22]:
# 삭제된 1을 다시 호출하면 Cache Miss 재발생
print("\n- 다시 heavy_calculation(1) 호출 (캐시에서 밀려났으므로 재연산):")
heavy_calculation(1)
print(f"   최종 캐시 상태: {heavy_calculation.cache_info()}")


- 다시 heavy_calculation(1) 호출 (캐시에서 밀려났으므로 재연산):
   최종 캐시 상태: CacheInfo(hits=34, misses=5, maxsize=3, currsize=3)


In [29]:
print("=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===")

for n in [5, 6, 7, 8]:
    print("\n1) heavy_calculation(1) 첫 번째 호출:")
    start_time = time.time()
    result = heavy_calculation(n)
    end_time = time.time()
    duration = end_time - start_time
    print(f"   캐시 상태: {heavy_calculation.cache_info()}")
    print(f"heavy_calculation(1) (캐시 히트) 호출에 실제 {duration:.4f}초가 소요되었습니다.")
    print('*'*100)

=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=44, misses=16, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=45, misses=16, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
   캐시 상태: CacheInfo(hits=46, misses=16, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
  [Cache Miss!] heavy_calculation(8) 실행 중... (3초 소요)
   캐시 상태: CacheInfo(hits=46, misses=17, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 3.0002초가 소요되었습니다.
****

In [30]:
print("=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===")

for n in [5, 6, 7]:
    print("\n1) heavy_calculation(1) 첫 번째 호출:")
    start_time = time.time()
    result = heavy_calculation(n)
    end_time = time.time()
    duration = end_time - start_time
    print(f"   캐시 상태: {heavy_calculation.cache_info()}")
    print(f"heavy_calculation(1) (캐시 히트) 호출에 실제 {duration:.4f}초가 소요되었습니다.")
    print('*'*100)

=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===

1) heavy_calculation(1) 첫 번째 호출:
  [Cache Miss!] heavy_calculation(5) 실행 중... (3초 소요)
   캐시 상태: CacheInfo(hits=46, misses=18, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 3.0002초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
  [Cache Miss!] heavy_calculation(6) 실행 중... (3초 소요)
   캐시 상태: CacheInfo(hits=46, misses=19, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 3.0002초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
  [Cache Miss!] heavy_calculation(7) 실행 중... (3초 소요)
   캐시 상태: CacheInfo(hits=46, misses=20, maxsize=3, currsize=3)
heavy_calculation(1) (캐시 히트) 호출에 실제 3.0002초가 소요되었습니다.
****************************************************************************************************


### cache 메모리안의 변수와 값을 확인

In [31]:
import time

class CustomCache:
    def __init__(self, capacity=3):
        self.capacity = capacity
        self.cache = {}  # 실제 캐시 저장소 (Dict)
        self.hits = 0
        self.misses = 0

    def __call__(self, n):
        # 1. Cache Hit
        if n in self.cache:
            self.hits += 1
            # LRU 효과: 최근 사용된 항목을 맨 뒤로 이동
            val = self.cache.pop(n)
            self.cache[n] = val
            return val

        # 2. Cache Miss
        self.misses += 1
        print(f"  [Miss!] heavy_calculation({n}) 실행 중...")
        time.sleep(1)
        result = n * 10

        # 용량 초과 시 가장 오래된 항목(첫 번째) 삭제 (LRU)
        if len(self.cache) >= self.capacity:
            oldest_key = next(iter(self.cache))
            del self.cache[oldest_key]
            print(f"  [LRU 삭제] 용량 초과로 인자 '{oldest_key}' 캐시에서 제거됨")

        self.cache[n] = result
        return result

    def show_info(self):
        print(f"\n[캐시 상태]")
        print(f" - Hit: {self.hits} | Miss: {self.misses}")
        print(f" - 현재 저장된 데이터 (Key: Value) -> {self.cache}")

# 사용예시
heavy_calc = CustomCache(capacity=3)

heavy_calc(1)
heavy_calc(2)
heavy_calc(3)
heavy_calc.show_info()  # {1: 10, 2: 20, 3: 30} 출력

heavy_calc(4)  # 1이 밀려나고 4가 들어감
heavy_calc.show_info()  # {2: 20, 3: 30, 4: 40} 출력

  [Miss!] heavy_calculation(1) 실행 중...
  [Miss!] heavy_calculation(2) 실행 중...
  [Miss!] heavy_calculation(3) 실행 중...

[캐시 상태]
 - Hit: 0 | Miss: 3
 - 현재 저장된 데이터 (Key: Value) -> {1: 10, 2: 20, 3: 30}
  [Miss!] heavy_calculation(4) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '1' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 4
 - 현재 저장된 데이터 (Key: Value) -> {2: 20, 3: 30, 4: 40}


In [32]:
for i in range(5, 10):
    heavy_calc(i)
    heavy_calc.show_info()

  [Miss!] heavy_calculation(5) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '2' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 5
 - 현재 저장된 데이터 (Key: Value) -> {3: 30, 4: 40, 5: 50}
  [Miss!] heavy_calculation(6) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '3' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 6
 - 현재 저장된 데이터 (Key: Value) -> {4: 40, 5: 50, 6: 60}
  [Miss!] heavy_calculation(7) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '4' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 7
 - 현재 저장된 데이터 (Key: Value) -> {5: 50, 6: 60, 7: 70}
  [Miss!] heavy_calculation(8) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '5' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 8
 - 현재 저장된 데이터 (Key: Value) -> {6: 60, 7: 70, 8: 80}
  [Miss!] heavy_calculation(9) 실행 중...
  [LRU 삭제] 용량 초과로 인자 '6' 캐시에서 제거됨

[캐시 상태]
 - Hit: 0 | Miss: 9
 - 현재 저장된 데이터 (Key: Value) -> {7: 70, 8: 80, 9: 90}


In [39]:
print("=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===")

for n in [10, 11, 12, 13, 10, 11, 12,]:
    print("\n1) heavy_calculation(1) 첫 번째 호출:")
    start_time = time.time()
    result = heavy_calc(n)
    end_time = time.time()
    duration = end_time - start_time
    print(f"   캐시 상태: {heavy_calc.show_info()}")
    print(f"heavy_calculation(1) (캐시 히트) 호출에 실제 {duration:.4f}초가 소요되었습니다.")
    print('*'*100)

=== 캐시 히트(Hit) & 미스(Miss) 동작 확인 ===

1) heavy_calculation(1) 첫 번째 호출:

[캐시 상태]
 - Hit: 9 | Miss: 29
 - 현재 저장된 데이터 (Key: Value) -> {11: 110, 12: 120, 10: 100}
   캐시 상태: None
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:

[캐시 상태]
 - Hit: 10 | Miss: 29
 - 현재 저장된 데이터 (Key: Value) -> {12: 120, 10: 100, 11: 110}
   캐시 상태: None
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:

[캐시 상태]
 - Hit: 11 | Miss: 29
 - 현재 저장된 데이터 (Key: Value) -> {10: 100, 11: 110, 12: 120}
   캐시 상태: None
heavy_calculation(1) (캐시 히트) 호출에 실제 0.0000초가 소요되었습니다.
****************************************************************************************************

1) heavy_calculation(1) 첫 번째 호출:
  [Miss!] heavy_calculation(13) 실행 중...
  [LRU 삭제] 용